# Paper-faithful LIFE on ISOT (human fake-vs-real) — negative control

Runs the **actual LIFE method, paper-faithful**, on **ISOT** (Fake.csv / True.csv), which is
**human-written on both sides**. Faithful = **LLaMA-2-7B reconstruction model** (not GPT-2) +
the **sigmoid + BCE head** (paper Eq 11–12), i.e. `train_bce.py` + `model_bce.py` — NOT the
BMES/CRF head.

This is a bigger, independent, different-domain replication of **§7j** (LLaMA-2-7B + BCE, HF-vs-HR
on PolitiFact++, where the faithful BCE head sat **at the majority prior** — LIFE detects
LLM-*generation*, not fakeness). Expected here: BCE **Acc ≈ majority baseline, low Macro-F1**.

**Labels:** Fake → `human_fake`, True → `human_true`. `train_bce.py` derives the binary label from
the suffix (`endswith('_fake')` → fake=1, real=0), so it needs **no edits** for ISOT.

**⚠️ Read the result carefully.** ISOT is **not source-matched**: ~99% of True.csv is Reuters
newswire, ~0% of Fake.csv is. So a **near-prior / low-F1** result cleanly supports *"LIFE can't
recognize human-written fake news"*; an **above-prior** result is **ambiguous** (LLaMA perplexity
may read newswire style, not veracity). The clean, topic-matched control remains HF-vs-HR (§7j).

**Scale:** 1,000 articles/class. **GPU required** — stage 1 (BERT) + stage 3 (LLaMA-2-7B, bf16
≈14 GB → T4/L4/A100). **5 stages:** 0 convert → 1 key sentences (BERT) → 2 merge → 3 LLaMA
perplexity features → 4 train the BCE head (+ multi-seed mean±std).

In [1]:
!pip install -q transformers datasets nltk tqdm sentencepiece accelerate  # torch preinstalled

In [2]:
import torch, nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to GPU')

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

PROJECT_DIR = '/content/drive/MyDrive/LIFE'
ISOT_DIR    = f'{PROJECT_DIR}/dataset/data/ISOT'          # holds Fake.csv / True.csv
WORK        = f'{ISOT_DIR}/isot_life'                     # all pipeline artifacts live here
CONVERTED   = f'{WORK}/converted'                         # stage 0 out: ISOT_fake/true.jsonl
FEATURES    = f'{WORK}/features_llama'                    # stage 3 out: LLaMA feature jsonl
KEYSENTS    = f'{WORK}/keysents.jsonl'                    # stage 1 out
BERT_CKPT   = f'{WORK}/isot_bert_model.pt'                # stage 1 key-sentence extractor
TRAIN_JSONL = f'{WORK}/train.jsonl'                       # stage 4 split (kept OUT of FEATURES
TEST_JSONL  = f'{WORK}/test.jsonl'                        #   so it is not re-ingested as input)

os.chdir(PROJECT_DIR)  # so `dataset/...` and `LIFE_train/...` script paths + imports resolve
os.makedirs(WORK, exist_ok=True)
print('ISOT Fake.csv found:', os.path.isfile(f'{ISOT_DIR}/Fake.csv'))
print('ISOT True.csv found:', os.path.isfile(f'{ISOT_DIR}/True.csv'))

ISOT Fake.csv found: True
ISOT True.csv found: True


In [6]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.7/644.7 kB 33.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.42.0 requires rich<14,>=12.4.4, but you have rich 11.2.0 which is incompatible.
pymc 5.28.5 requires rich>=13.7.1, but you have rich 11.2.0 which is incompatible.


## Stage 0 — convert ISOT CSV → JSONL, subsample 1,000 / class
Fake → `human_fake`, True → `human_true`; synthesizes ids (ISOT has none).

In [16]:
!python dataset/0_convert_isot.py --input_dir "{ISOT_DIR}" --output_dir "{CONVERTED}" --sample_size 1000 --seed 42

Fake.csv -> ISOT_fake.jsonl: 1000 records (label=human_fake)
True.csv -> ISOT_true.jsonl: 1000 records (label=human_true)


## Stage 1 — key-sentence extraction (BERT)
Trains `bert-base-uncased` to separate fake/true, then leave-one-sentence-out to pick the top-10
sentences per article (~10–20 min on a T4 for 2k articles). *On ISOT the BERT separates the
classes easily via the Reuters artifact — but it only **selects which sentences get scored**;
LIFE's verdict comes from the LLaMA perplexity features + BCE head in stages 3–4.*

In [17]:
!python dataset/1_keySentenceExtraction.py --data_dir "{CONVERTED}" --output_file "{KEYSENTS}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

config.json: 100% 570/570 [00:00<00:00, 2.25MB/s]

model.safetensors: downloading bytes:  70% 309M/440M [00:01<00:00, 319MB/s, 22.2MB/s  ]
model.safetensors: downloading bytes:  94% 415M/440M [00:02<00:00, 386MB/s, 35.1MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 170MB/s, 38.9MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 180MB/s, 41.6MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 4327.84it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  |

## Stage 2 — merge the key sentences back into the article JSONL
In-place: adds a `sentence` field to `CONVERTED/ISOT_fake.jsonl` and `ISOT_true.jsonl`.

In [18]:
!python dataset/2_concate.py --folder_path "{CONVERTED}" --important_sentences_file "{KEYSENTS}"

所有 .jsonl 文件已成功更新。


## Stage 3 — LLaMA-2-7B perplexity features (paper-faithful reconstruction model)
The paper's reconstruction mLLM is **LLaMA-2-7B** (§4.1.3), scored with the SentencePiece
`--scorer llama` path in `backend_utils.SPLlamaTokenizerPPLCalc`. `NousResearch/Llama-2-7b-hf`
is an **ungated mirror** of the official weights → **no HF token needed** (swap to
`meta-llama/Llama-2-7b-hf` if you have Meta approval). bf16 ≈14 GB; ~15–40 min for 2k articles.
First run downloads ~13 GB of weights.

In [7]:
!python dataset/3_gen_features_local.py --input_dir "{CONVERTED}" --output_dir "{FEATURES}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

Streaming output truncated to the last 5000 lines.
455 484
486 527
598 631
633 664
666 702
758 770
854 908
0 117
188 241
243 261
397 441
506 518
521 553
555 569
571 603
605 665
667 725
727 775
 44% 442/1000 [00:35<00:47, 11.69it/s]0 57
58 168
170 256
258 284
286 552
554 606
610 644
646 690
0 117
177 241
244 294
384 424
426 454
457 501
504 535
540 612
616 650
653 671
673 726
 44% 444/1000 [00:35<00:44, 12.45it/s]0 117
271 319
322 380
382 418
420 440
575 579
581 597
599 655
657 665
667 705
707 735
0 57
58 138
140 187
189 251
253 267
 45% 446/1000 [00:36<00:41, 13.50it/s]0 57
58 146
149 179
181 221
224 321
324 367
0 57
58 116
119 137
139 178
0 57
58 130
132 214
216 276
 45% 449/1000 [00:36<00:34, 15.87it/s]0 117
196 266
400 460
462 492
494 546
598 648
650 702
736 782
936 952
954 972
974 974
0 117
196 265
269 318
321 335
338 382
384 410
414 483
486 516
519 571
575 641
700 779
 45% 451/1000 [00:36<00:37, 14.80it/s]0 57
58 104
106 112
114 164
166 172
174 204
0 117
118 182
184 230
232 262
264

## Stage 4 — train the paper BCE head (sigmoid + binary cross-entropy)
`train_bce.py` / `model_bce.py`: masked mean-pool → one sigmoid probability per article, BCE loss
(fake=1, real=0), evaluated at article level — no BMES/CRF/majority-vote. `--split_dataset` uses
the deterministic seed-0 train/test split. **Watch Acc vs the majority baseline and Macro-F1:**
at-prior + low F1 → LIFE has no fingerprint for human-written fake news (the point of this run).
`real = human_true`, `fake = human_fake`.

In [9]:
!python LIFE_train/train_bce.py --split_dataset --data_path "{FEATURES}" --train_path "{TRAIN_JSONL}" --test_path "{TEST_JSONL}" --num_train_epochs 10 --seed 0

Log INFO: split dataset...
********************************
The overall data sources:
['ISOT_fake.jsonl', 'ISOT_true.jsonl']
100% 1600/1600 [00:00<00:00, 3202.76it/s]
100% 400/400 [00:00<00:00, 3139.61it/s]

The number of train dataset: 1600
The number of test  dataset: 400
********************************
100% 1600/1600 [00:01<00:00, 1258.26it/s]
100% 400/400 [00:00<00:00, 7665.90it/s]
seed: 0
--------------------------------BCE head (paper Eq 11-12)--------------------------------
Log INFO: do train...
Epoch:   0% 0/10 [00:00<?, ?it/s]
Iteration:   0% 0/50 [00:00<?, ?it/s]
Iteration:   2% 1/50 [00:01<00:55,  1.13s/it]
Iteration:   4% 2/50 [00:01<00:31,  1.50it/s]
Iteration:   6% 3/50 [00:01<00:19,  2.44it/s]
Iteration:   8% 4/50 [00:01<00:13,  3.46it/s]
Iteration:  10% 5/50 [00:01<00:09,  4.51it/s]
Iteration:  12% 6/50 [00:01<00:08,  5.48it/s]
Iteration:  14% 7/50 [00:01<00:06,  6.32it/s]
Iteration:  16% 8/50 [00:02<00:05,  7.08it/s]
Iteration:  18% 9/50 [00:02<00:05,  7.77it/s]
Iter

### Stage 4b — multi-seed mean ± std (the honest read for a negative control)
The split is fixed at seed 0; only model init + batch order vary. §7j showed single seeds are
misleading here — some collapse to "always real", a few catch a handful of fakes — so report
**mean ± std** over seeds. Features are tiny → this is a couple of minutes. Extend `range(10)` to
`range(21)` to match the paper's 0–20 sweep.

In [10]:
import subprocess, re, numpy as np
accs, f1s = [], []
for seed in range(10):
    out = subprocess.run(
        ['python', 'LIFE_train/train_bce.py', '--split_dataset',
         '--data_path', FEATURES, '--train_path', TRAIN_JSONL, '--test_path', TEST_JSONL,
         '--num_train_epochs', '10', '--seed', str(seed)],
        capture_output=True, text=True).stdout
    a = re.findall(r'Accuracy: ([\d.]+)', out)          # printed every epoch; take the last
    f = re.findall(r'Macro F1 Score: ([\d.]+)', out)
    if a and f:
        accs.append(float(a[-1])); f1s.append(float(f[-1]))
        print(f'seed {seed:>2}: Acc {a[-1]:>5}  Macro-F1 {f[-1]:>5}')
    else:
        print(f'seed {seed:>2}: FAILED (no metrics parsed)')
print(f'\nBCE head over {len(accs)} seeds:  '
      f'Acc {np.mean(accs):.2f} ± {np.std(accs):.2f}   '
      f'Macro-F1 {np.mean(f1s):.2f} ± {np.std(f1s):.2f}')

seed  0: Acc  76.2  Macro-F1  76.2
seed  1: Acc  74.2  Macro-F1  74.2
seed  2: Acc  74.0  Macro-F1  74.0
seed  3: Acc  71.2  Macro-F1  71.2
seed  4: Acc  77.2  Macro-F1  77.2
seed  5: Acc  78.0  Macro-F1  78.0
seed  6: Acc  75.8  Macro-F1  75.7
seed  7: Acc  71.8  Macro-F1  71.7
seed  8: Acc  77.2  Macro-F1  77.2
seed  9: Acc  75.0  Macro-F1  75.0

BCE head over 10 seeds:  Acc 75.06 ± 2.17   Macro-F1 75.04 ± 2.18


## Notes
- **What success looks like:** BCE Acc ≈ majority baseline (~50% at 1:1) with low Macro-F1, and a
  tight Acc std / noisy F1 std (the prior-collapse signature from §7j) → confirms LIFE's fingerprint
  can't separate human fake from human real. An above-prior number is **not** a refutation — see the
  Reuters-artifact caveat in the header.
- **Sync to Drive before running:** `dataset/0_convert_isot.py` (new). Everything else is reused
  UNCHANGED and should already be on Drive from your MF-vs-MR / §7j runs:
  `dataset/1_keySentenceExtraction.py`, `2_concate.py`, `3_gen_features_local.py`, `backend_utils.py`,
  `LIFE_train/train_bce.py`, `LIFE_train/model_bce.py`, `LIFE_train/dataloader.py`, `model.py`.
- **1:1 class balance:** this run uses 1,000 fake / 1,000 real, so the majority baseline is ~50%
  (cleaner to read than §7j's 2:1 HF/HR, where the prior was ~67%).
- **Change scale:** edit `--sample_size` in Stage 0 (0 = all ~45k). **Re-run cleanly:** delete `WORK`
  first; Stage 1 *loads* `BERT_CKPT` if it exists instead of retraining.
- **Compare vs the released BMES/CRF head** (the §7j A-side): run `LIFE_train/train.py` on the same
  `FEATURES` — but note that head is nearly insensitive to feature quality (§7h/§7i), so its number
  is a mechanism artifact, not a fingerprint signal. The BCE head is the honest instrument.